# Regresión lineal desde cero: del avión al código

Este cuaderno acompaña al **capítulo 5** del curso de aprendizaje automático desde cero. Aquí armamos, con código de verdad, todo lo que vimos en los capítulos 1 a 4:

1. **Los datos**: 14 vuelos (peso del avión → consumo de combustible).
2. **La recta y la pérdida** (capítulos 1 y 2).
3. **El descenso de gradiente, escrito a mano con NumPy** (capítulo 3).
4. **Los hiperparámetros**: tasa de aprendizaje, tamaño del lote y épocas (capítulo 4).
5. **Lo mismo con Keras**, en pocas líneas.
6. **Usar el modelo** para predecir.

Ejecuta las celdas en orden (Mayús + Intro). Los datos son **sintéticos**: inventados para aprender, no de una aerolínea real.

## 1. Los datos

`x` es el peso del avión al despegar, en toneladas. `y` es lo que consumió en crucero, en litros por hora.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([58.2, 59.5, 62.0, 63.8, 66.5, 67.4, 69.8, 71.5, 73.0, 75.2, 77.0, 78.5, 80.2, 82.0])   # toneladas
y = np.array([2310, 2420, 2390, 2540, 2520, 2680, 2640, 2810, 2770, 2930, 2890, 3060, 3040, 3190], dtype=float)   # litros por hora

print("vuelos:", len(x))
print("peso:    de", x.min(), "a", x.max(), "toneladas")
print("consumo: de", y.min(), "a", y.max(), "L/h")

In [ ]:
plt.scatter(x, y, color="gray")
plt.xlabel("x₁ · peso del avión (t)"); plt.ylabel("y · consumo (L/h)")
plt.title("Los 14 vuelos"); plt.show()

## 2. La recta y la pérdida

El modelo es una recta: **y′ = b + w₁ · x₁**. `w` es el peso (la pendiente) y `b` es el sesgo.

La pérdida que usamos es el **MSE**: el promedio de los errores al cuadrado.

In [ ]:
def predecir(x, w, b):
    return b + w * x

def mse(w, b):
    error = predecir(x, w, b) - y
    return np.mean(error ** 2)

# una recta mala y la recta buena del curso
print("recta mala   (w = 26,   b = 294):  MSE =", round(mse(26, 294)))
print("recta buena  (w = 34.6, b = 294):  MSE =", round(mse(34.6, 294)))

## 3. El descenso de gradiente, a mano

La receta del capítulo 3:

1. Calcula la pérdida con el peso y el sesgo de ahora.
2. Calcula hacia dónde baja la pérdida (las dos pendientes: la del peso y la del sesgo).
3. Mueve el peso y el sesgo un poco en esa dirección. Ese "poco" lo fija la **tasa de aprendizaje**.
4. Repite.

**Un detalle práctico.** Antes de entrenar, restamos 70 toneladas a cada peso (`x_c = x − 70`). La recta que sale es la misma, pero así el descenso llega mucho más rápido. Al final devolvemos el sesgo a toneladas reales. Más abajo puedes ver qué pasa si no lo haces.

In [ ]:
CENTRO = 70.0
x_c = x - CENTRO          # toneladas por encima o por debajo de 70

def entrenar(tasa, epocas, lote=None, w=5.0, b_c=2400.0, semilla=7):
    """Descenso de gradiente. lote=None usa los 14 vuelos en cada paso (lote completo).
    Devuelve w, b (en toneladas reales) y la pérdida después de cada paso."""
    n = len(x_c)
    lote = n if lote is None else lote
    azar = np.random.default_rng(semilla)
    historia = [np.mean((b_c + w * x_c - y) ** 2)]
    for _ in range(epocas):                      # una época = una pasada por todos los vuelos
        orden = azar.permutation(n)
        for k in range(0, n, lote):              # un paso por cada lote
            i = orden[k:k + lote]
            error = b_c + w * x_c[i] - y[i]      # predicho − real
            pend_w = np.mean(2 * error * x_c[i]) # pendiente de la pérdida según el peso
            pend_b = np.mean(2 * error)          # pendiente de la pérdida según el sesgo
            w   = w   - tasa * pend_w            # paso = tasa × pendiente, en contra
            b_c = b_c - tasa * pend_b
            historia.append(np.mean((b_c + w * x_c - y) ** 2))
    return w, b_c - CENTRO * w, np.array(historia)

In [ ]:
# hiperparámetros: los elegimos nosotros
TASA   = 0.006
EPOCAS = 600

w, b, perdida = entrenar(TASA, EPOCAS)   # lote completo: 14 vuelos por paso

for epoca in [0, 1, 10, 100, 200, 400, 600]:
    print(f"época {epoca:>3}:  MSE = {perdida[epoca]:>9,.0f}")
print()
print(f"peso  w = {w:.2f}")
print(f"sesgo b = {b:.1f}")
print(f"MSE = {mse(w, b):,.0f}   →   RMSE = {mse(w, b) ** 0.5:.1f} L/h")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(perdida, color="purple"); a1.set_xlabel("épocas"); a1.set_ylabel("pérdida (MSE)"); a1.set_title("Curva de pérdida")
a2.scatter(x, y, color="gray"); a2.plot(x, predecir(x, w, b), color="black")
a2.set_xlabel("x₁ · peso del avión (t)"); a2.set_ylabel("y · consumo (L/h)"); a2.set_title(f"y′ = {b:.0f} + {w:.1f} · x₁")
plt.show()

**Comprobación.** Para una recta existe una fórmula exacta (mínimos cuadrados). El descenso de gradiente tiene que llegar al mismo sitio:

In [ ]:
w_exacto, b_exacto = np.polyfit(x, y, 1)
print(f"fórmula exacta:        w = {w_exacto:.2f}   b = {b_exacto:.1f}")
print(f"descenso de gradiente: w = {w:.2f}   b = {b:.1f}")

## 4. Jugar con los hiperparámetros

Lo del capítulo 4, ahora con tus manos. Cambia los números y vuelve a ejecutar.

### 4.1 La tasa de aprendizaje

In [ ]:
_, _, chica  = entrenar(tasa=0.0006, epocas=600)   # diez veces más chica
_, _, buena  = entrenar(tasa=0.006,  epocas=600)
_, _, grande = entrenar(tasa=0.02,   epocas=8)     # más de tres veces más grande

print("tasa 0.0006 → MSE tras 600 épocas:", round(chica[-1]), "  (el fondo es", round(buena[-1]), ")")
print("tasa 0.006  → MSE tras 600 épocas:", round(buena[-1]))
print("tasa 0.02   → MSE en las primeras épocas:", [round(v) for v in grande[:5]], "… ¡sube!")

plt.plot(buena, label="0.006 (buena)", color="purple"); plt.plot(chica, label="0.0006 (muy chica)", color="gray")
plt.xlabel("épocas"); plt.ylabel("pérdida (MSE)"); plt.legend(); plt.title("La misma receta, otra tasa"); plt.show()

### 4.2 El tamaño del lote

Con la misma tasa y 100 épocas: lote completo (14), mini-lotes de 7 y de uno en uno.

In [ ]:
resultados = {}
for nombre, lote in [("lote completo (14)", 14), ("mini-lote (7)", 7), ("de uno en uno (1)", 1)]:
    w_l, b_l, h = entrenar(tasa=0.006, epocas=100, lote=lote)
    subidas = np.sum(h[1:] > h[:-1])
    resultados[nombre] = h
    print(f"{nombre:<20} pasos: {len(h) - 1:>5}   la pérdida subió en {subidas:>3} pasos ({100 * subidas / (len(h) - 1):.0f} %)   MSE final: {h[-1]:>7,.0f}")

for nombre, h in resultados.items():
    plt.plot(np.linspace(0, 100, len(h)), h, label=nombre, linewidth=1)
plt.ylim(0, 170000); plt.xlabel("épocas"); plt.ylabel("pérdida (MSE)"); plt.legend(); plt.title("El ruido de los lotes pequeños"); plt.show()

### 4.3 ¿Y si no restamos las 70 toneladas?

Con las toneladas tal cual, hay que usar una tasa muchísimo más chica para que no se dispare, y el sesgo casi no se mueve:

In [ ]:
w_s, b_s = 5.0, 2400.0 - 70 * 5.0     # la misma recta de partida, en toneladas reales
for _ in range(600):
    error = b_s + w_s * x - y
    w_s, b_s = w_s - 0.0001 * np.mean(2 * error * x), b_s - 0.0001 * np.mean(2 * error)
print(f"sin centrar, 600 épocas:  w = {w_s:.2f}   b = {b_s:.1f}   MSE = {mse(w_s, b_s):,.0f}")
print(f"centrando,   600 épocas:  w = {w:.2f}   b = {b:.1f}   MSE = {mse(w, b):,.0f}")

## 5. Lo mismo con Keras

En la práctica casi nadie escribe el descenso de gradiente a mano: se usa una biblioteca. **Keras** es una de las más conocidas. Fíjate que aparecen las mismas piezas:

| Lo que vimos | En Keras |
|---|---|
| la recta (un peso y un sesgo) | `Dense(units=1)` |
| la pérdida | `loss="mean_squared_error"` |
| el descenso de gradiente y la tasa | `SGD(learning_rate=0.006)` |
| el lote y las épocas | `batch_size=14, epochs=600` |

Para comparar de verdad, le damos la misma recta de partida que usamos a mano (peso 5, sesgo 2400).

In [ ]:
import keras

modelo = keras.Sequential([
    keras.Input(shape=(1,)),
    keras.layers.Dense(units=1,                                   # una recta: un peso y un sesgo
                       kernel_initializer=keras.initializers.Constant(5.0),
                       bias_initializer=keras.initializers.Constant(2400.0)),
])
modelo.compile(loss="mean_squared_error",                           # la pérdida: MSE
               optimizer=keras.optimizers.SGD(learning_rate=0.006)) # descenso de gradiente + tasa

registro = modelo.fit(x_c, y, batch_size=14, epochs=600, verbose=0)  # lote y épocas

w_k = float(modelo.layers[0].get_weights()[0][0, 0])
b_k = float(modelo.layers[0].get_weights()[1][0]) - CENTRO * w_k
print(f"a mano (NumPy):  w = {w:.2f}   b = {b:.1f}   MSE = {mse(w, b):,.0f}")
print(f"con Keras:       w = {w_k:.2f}   b = {b_k:.1f}   MSE = {mse(w_k, b_k):,.0f}")

In [ ]:
plt.plot(perdida[:-1], label="a mano (NumPy)", color="purple", linewidth=4, alpha=0.4)
plt.plot(registro.history["loss"], label="Keras", color="black", linewidth=1)
plt.xlabel("épocas"); plt.ylabel("pérdida (MSE)"); plt.legend(); plt.title("Las dos curvas de pérdida, una encima de la otra"); plt.show()

**Tu turno.** Cambia `learning_rate`, `batch_size` o `epochs` en la celda de Keras y vuelve a ejecutarla. ¿Qué pasa con `learning_rate=0.02`? ¿Y con `batch_size=1`?

## 6. Usar el modelo

El modelo ya está entrenado. Le damos un peso y nos devuelve el consumo que espera.

In [ ]:
toneladas = 73
print(f"un avión de {toneladas} t  →  {predecir(toneladas, w, b):,.0f} L/h")
print("(ese vuelo está en los datos: consumió 2,770 L/h, así que el modelo se pasó por unos 50)")

Y la comprobación del final: lo que predice para cada uno de los 14 vuelos, al lado de lo que pasó de verdad.

In [ ]:
predicho = predecir(x, w, b)
print(" toneladas   predicho   real   diferencia (predicho − real)")
for xi, yi, pi in zip(x, y, predicho):
    print(f"     {xi:>5.1f}     {pi:>6.0f}   {yi:>5.0f}    {pi - yi:>+5.0f}")
print()
print(f"error medio, sin signo (MAE): {np.mean(np.abs(y - predicho)):.0f} L/h")

### Ojo

Aquí comprobamos el modelo con los **mismos vuelos** con los que lo entrenamos. Sirve para aprender, pero en un proyecto real se guarda una parte de los datos **sin usarla** en el entrenamiento, para probar el modelo con vuelos que nunca vio.

---
*Datos sintéticos, con fines educativos.*